# Reviewer Notebook

This notebook is a lightweight front-end for `run.py`.
Edit the configuration values in the next cell, then run the cells in order.

- Main methods supported here: `boost`, `fixed`, `static`, `random_search`, `best_utility`, `botorch_default`, `boost_fixedhp`, `random_kernel_af`
- `No-PASt-BO` and `HEBO` remain separate because they use different dependency stacks

In [ ]:
# Edit only this cell for most reviewer-side runs.

METHOD = "boost"              # boost | fixed | static | random_search | best_utility | botorch_default | boost_fixedhp | random_kernel_af
SPACE = "synthetic"           # synthetic | hpob

# Use OBJECTIVE when SPACE == "synthetic"
OBJECTIVE = "ackley"          # ackley | levy | rosenbrock | all

# Use HPOB_TASK when SPACE == "hpob"
HPOB_TASK = "5891_8D"         # 2277_15D, 5636_6D, ..., 6794_10D, or all

# Required only for METHOD == "fixed"
KERNEL = None                  # Matern32 | Matern52 | RBF | RQ
ACQUISITION = None             # EI | PI | UCB | PM

TRIALS = 1
MAX_ITER = 100 # including initial points
N_INIT_POINTS = 10
DEVICE = "cpu"                 # paper used CPU for all methods except botorch_default.
                               # For botorch_default, set to "cuda" (its published results
                               # were obtained on GPU; CPU is roughly 10x slower).
OUTPUT_DIR = "tests/notebook_demo"
DRY_RUN = False

In [ ]:
from pathlib import Path
import shlex
import sys

ROOT = Path.cwd()
RUN_FILE = ROOT / "run.py"

if not RUN_FILE.exists():
    raise FileNotFoundError("Open this notebook from the repository root, where run.py is located.")

VALID_METHODS = {"boost", "fixed", "static", "random_search", "best_utility", "botorch_default", "boost_fixedhp", "random_kernel_af"}
VALID_SPACES = {"synthetic", "hpob"}
if METHOD not in VALID_METHODS:
    raise ValueError(f"Unsupported METHOD: {METHOD}")
if SPACE not in VALID_SPACES:
    raise ValueError(f"Unsupported SPACE: {SPACE}")
if METHOD == "fixed" and (not KERNEL or not ACQUISITION):
    raise ValueError("METHOD='fixed' requires both KERNEL and ACQUISITION.")
    raise ValueError("MAX_ITER must be greater than N_INIT_POINTS.")

cmd = [
    sys.executable,
    str(RUN_FILE),
    "--method", METHOD,
    "--space", SPACE,
    "--trials", str(TRIALS),
    "--max-iter", str(MAX_ITER),
    "--n-init-points", str(N_INIT_POINTS),
    "--device", DEVICE,
]

if SPACE == "synthetic":
    cmd += ["--objective", OBJECTIVE]
else:
    cmd += ["--hpob-task", HPOB_TASK]

if KERNEL:
    cmd += ["--kernel", KERNEL]
if ACQUISITION:
    cmd += ["--acquisition", ACQUISITION]
if OUTPUT_DIR:
    cmd += ["--output-dir", OUTPUT_DIR]
if DRY_RUN:
    cmd += ["--dry-run"]

print("Prepared command:")
print(shlex.join(cmd))

In [ ]:
import subprocess

subprocess.run(cmd, check=True)

## Notes

- `boost`, `static`, `random_search`, `botorch_default`, `boost_fixedhp`, and `random_kernel_af` follow the default `TBD` placeholders used by the original scripts unless you override them.
- `best_utility` defaults to `EI` inside `run.py` if no acquisition is provided.
- `botorch_default` is much faster on GPU (`DEVICE = "cuda"`) than on CPU; the published `botorch_default` results were obtained on GPU.
- `No-PASt-BO` and `HEBO` are not routed through this notebook. Use `Code/Adaptive_Acquisition/nopastbo_*.py` and `Code/HEBO/Test_*_hebo.py` in their separate environments.